In [ ]:
# \\wsl.localhost\Ubuntu-18.04\home\shiv1901\safeil-data-collection-main\safeil-data-collection-main\SafeDICE\dataset\safetygym\ppo_lagrangian_PointGoal1_s0.pickle

In [12]:
import pickle
filepath_ppolag = r'\\wsl.localhost\Ubuntu-18.04\home\shiv1901\safeil-data-collection-main\safeil-data-collection-main\SafeDICE\dataset\safetygym\ppo_lagrangian_PointGoal1_s0.pickle'
filepath_ppo= r'\\wsl.localhost\Ubuntu-18.04\home\shiv1901\safeil-data-collection-main\safeil-data-collection-main\SafeDICE\dataset\safetygym\ppo_PointGoal1_s0.pickle'
with open(filepath_ppolag, 'rb') as f:
    data_ppolag = pickle.load(f)
with open(filepath_ppo, 'rb') as f:
    data_ppo = pickle.load(f)

C:\Users\shivp\AppData\Local\Temp\ipykernel_27752\1289883611.py:5: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data_ppolag = pickle.load(f)
C:\Users\shivp\AppData\Local\Temp\ipykernel_27752\1289883611.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to acces

In [16]:
import numpy as np

def inspect_cumulative_reward_per_trajectory(loaded):
    def pick(mapping, keys):
        for k in keys:
            if isinstance(mapping, dict) and k in mapping:
                return mapping[k]
        return None

    payload = loaded
    if isinstance(payload, dict):
        ts = payload.get("training_state", None)
        if isinstance(ts, dict):
            for k in ["dataset", "data", "replay_buffer", "buffer", "expert_data", "trajectories"]:
                if k in ts:
                    payload = ts[k]
                    break
        if payload is loaded:
            for k in ["dataset", "data", "expert_data", "trajectories"]:
                if k in payload:
                    payload = payload[k]
                    break

    if isinstance(payload, list):
        traj_returns = []
        for traj in payload:
            if isinstance(traj, dict):
                r = pick(traj, ["rewards", "reward", "rews", "r"])
                if r is not None:
                    r = np.asarray(r, dtype=np.float32).reshape(-1)
                    if len(r) > 0:
                        traj_returns.append(float(r.sum()))
        if len(traj_returns) == 0:
            raise ValueError("Could not find rewards in list-style trajectories.")

        print("Num trajectories:", len(traj_returns))
        print("Cumulative reward per trajectory (first 10):", np.round(traj_returns[:10], 4))
        print("Mean cumulative reward:", float(np.mean(traj_returns)))
        print("Std cumulative reward:", float(np.std(traj_returns)))
        print("Min cumulative reward:", float(np.min(traj_returns)))
        print("Max cumulative reward:", float(np.max(traj_returns)))
        return traj_returns

    if not isinstance(payload, dict):
        raise TypeError("Unsupported dataset type. Expected dict or list.")

    rewards = pick(payload, ["rewards", "reward", "rews", "r", "env_rewards"])
    if rewards is None:
        raise KeyError(f"Could not find rewards. Available keys: {list(payload.keys())}")
    rewards = np.asarray(rewards, dtype=np.float32).reshape(-1)

    dones = pick(payload, ["dones", "done", "terminals", "terminal", "episode_ends", "timeouts"])
    if dones is None:
        total = float(rewards.sum())
        print("No done flags found. Returning single cumulative reward over all transitions.")
        print("Total cumulative reward:", total)
        print("Min cumulative reward:", total)
        print("Max cumulative reward:", total)
        return [total]

    dones = np.asarray(dones).reshape(-1).astype(bool)
    n = min(len(rewards), len(dones))
    rewards, dones = rewards[:n], dones[:n]

    traj_returns = []
    start = 0
    for i, d in enumerate(dones):
        if d:
            seg = rewards[start:i + 1]
            if len(seg) > 0:
                traj_returns.append(float(seg.sum()))
            start = i + 1
    if start < len(rewards):
        seg = rewards[start:]
        if len(seg) > 0:
            traj_returns.append(float(seg.sum()))

    print("Num trajectories:", len(traj_returns))
    print("Cumulative reward per trajectory (first 10):", np.round(traj_returns[:10], 4))
    print("Mean cumulative reward:", float(np.mean(traj_returns)))
    print("Std cumulative reward:", float(np.std(traj_returns)))
    print("Min cumulative reward:", float(np.min(traj_returns)))
    print("Max cumulative reward:", float(np.max(traj_returns)))
    return traj_returns


def inspect_cumulative_cost_per_trajectory(loaded):
    def pick(mapping, keys):
        for k in keys:
            if isinstance(mapping, dict) and k in mapping:
                return mapping[k]
        return None

    payload = loaded
    if isinstance(payload, dict):
        ts = payload.get("training_state", None)
        if isinstance(ts, dict):
            for k in ["dataset", "data", "replay_buffer", "buffer", "expert_data", "trajectories"]:
                if k in ts:
                    payload = ts[k]
                    break
        if payload is loaded:
            for k in ["dataset", "data", "expert_data", "trajectories"]:
                if k in payload:
                    payload = payload[k]
                    break

    if isinstance(payload, list):
        traj_cost_sums = []
        for traj in payload:
            if isinstance(traj, dict):
                c = pick(traj, ["costs", "cost", "c"])
                if c is not None:
                    c = np.asarray(c, dtype=np.float32).reshape(-1)
                    if len(c) > 0:
                        traj_cost_sums.append(float(c.sum()))

        if len(traj_cost_sums) == 0:
            raise ValueError("Could not find costs in list-style trajectories.")

        print("Num trajectories:", len(traj_cost_sums))
        print("Cumulative cost per trajectory (first 10):", np.round(traj_cost_sums[:10], 4))
        print("Mean cumulative cost:", float(np.mean(traj_cost_sums)))
        print("Std cumulative cost:", float(np.std(traj_cost_sums)))
        print("Min cumulative cost:", float(np.min(traj_cost_sums)))
        print("Max cumulative cost:", float(np.max(traj_cost_sums)))
        return traj_cost_sums

    if not isinstance(payload, dict):
        raise TypeError("Unsupported dataset type. Expected dict or list.")

    costs = pick(payload, ["costs", "cost", "c"])
    if costs is None:
        raise KeyError(f"Could not find costs. Available keys: {list(payload.keys())}")
    costs = np.asarray(costs, dtype=np.float32).reshape(-1)

    dones = pick(payload, ["dones", "done", "terminals", "terminal", "episode_ends", "timeouts"])
    if dones is None:
        total = float(costs.sum())
        print("No done flags found. Returning single cumulative cost over all transitions.")
        print("Total cumulative cost:", total)
        print("Min cumulative cost:", total)
        print("Max cumulative cost:", total)
        return [total]

    dones = np.asarray(dones).reshape(-1).astype(bool)
    n = min(len(costs), len(dones))
    costs, dones = costs[:n], dones[:n]

    traj_cost_sums = []
    start = 0
    for i, d in enumerate(dones):
        if d:
            seg = costs[start:i + 1]
            if len(seg) > 0:
                traj_cost_sums.append(float(seg.sum()))
            start = i + 1
    if start < len(costs):
        seg = costs[start:]
        if len(seg) > 0:
            traj_cost_sums.append(float(seg.sum()))

    print("Num trajectories:", len(traj_cost_sums))
    print("Cumulative cost per trajectory (first 10):", np.round(traj_cost_sums[:10], 4))
    print("Mean cumulative cost:", float(np.mean(traj_cost_sums)))
    print("Std cumulative cost:", float(np.std(traj_cost_sums)))
    print("Min cumulative cost:", float(np.min(traj_cost_sums)))
    print("Max cumulative cost:", float(np.max(traj_cost_sums)))
    return traj_cost_sums

In [17]:
# def inspect_cumulative_cost_per_trajectory(loaded):
#     def pick(mapping, keys):
#         for k in keys:
#             if isinstance(mapping, dict) and k in mapping:
#                 return mapping[k]
#         return None

#     payload = loaded
#     if isinstance(payload, dict):
#         ts = payload.get("training_state", None)
#         if isinstance(ts, dict):
#             for k in ["dataset", "data", "replay_buffer", "buffer", "expert_data", "trajectories"]:
#                 if k in ts:
#                     payload = ts[k]
#                     break

#         if payload is loaded:
#             for k in ["dataset", "data", "expert_data", "trajectories"]:
#                 if k in payload:
#                     payload = payload[k]
#                     break

#     # Case A: list of trajectory dicts
#     if isinstance(payload, list):
#         traj_cost_sums = []
#         for traj in payload:
#             if isinstance(traj, dict):
#                 c = pick(traj, ["costs", "cost", "c"])
#                 if c is not None:
#                     c = np.asarray(c, dtype=np.float32).reshape(-1)
#                     if len(c) > 0:
#                         traj_cost_sums.append(float(c.sum()))

#         if len(traj_cost_sums) == 0:
#             raise ValueError("Could not find costs in list-style trajectories.")

#         print("Num trajectories:", len(traj_cost_sums))
#         print("Cumulative cost per trajectory (first 10):", np.round(traj_cost_sums[:10], 4))
#         print("Mean cumulative cost:", float(np.mean(traj_cost_sums)))
#         print("Std cumulative cost:", float(np.std(traj_cost_sums)))
#         return traj_cost_sums

#     # Case B: flat transitions dict + done flags
#     if not isinstance(payload, dict):
#         raise TypeError("Unsupported dataset type. Expected dict or list.")

#     costs = pick(payload, ["costs", "cost", "c"])
#     if costs is None:
#         raise KeyError(f"Could not find costs. Available keys: {list(payload.keys())}")
#     costs = np.asarray(costs, dtype=np.float32).reshape(-1)

#     dones = pick(payload, ["dones", "done", "terminals", "terminal", "episode_ends", "timeouts"])
#     if dones is None:
#         print("No done flags found. Returning single cumulative cost over all transitions.")
#         total = float(costs.sum())
#         print("Total cumulative cost:", total)
#         return [total]

#     dones = np.asarray(dones).reshape(-1).astype(bool)
#     n = min(len(costs), len(dones))
#     costs, dones = costs[:n], dones[:n]

#     traj_cost_sums = []
#     start = 0
#     for i, d in enumerate(dones):
#         if d:
#             seg = costs[start:i + 1]
#             if len(seg) > 0:
#                 traj_cost_sums.append(float(seg.sum()))
#             start = i + 1

#     if start < len(costs):
#         seg = costs[start:]
#         if len(seg) > 0:
#             traj_cost_sums.append(float(seg.sum()))

#     print("Num trajectories:", len(traj_cost_sums))
#     print("Cumulative cost per trajectory (first 10):", np.round(traj_cost_sums[:10], 4))
#     print("Mean cumulative cost:", float(np.mean(traj_cost_sums)))
#     print("Std cumulative cost:", float(np.std(traj_cost_sums)))
#     return traj_cost_sums

In [18]:
print("Inspecting PPO Lagrangian dataset:")
traj_costs_ppolag = inspect_cumulative_cost_per_trajectory(data_ppolag)
traj_rewards_ppolag = inspect_cumulative_reward_per_trajectory(data_ppolag)
print("\nInspecting PPO dataset:")
traj_costs_ppo = inspect_cumulative_cost_per_trajectory(data_ppo)
traj_rewards_ppo = inspect_cumulative_reward_per_trajectory(data_ppo)

Inspecting PPO Lagrangian dataset:
Num trajectories: 1000
Cumulative cost per trajectory (first 10): [ 5.  0.  0.  0. 33.  8.  0. 16. 29. 25.]
Mean cumulative cost: 24.718
Std cumulative cost: 22.471993147026367
Min cumulative cost: 0.0
Max cumulative cost: 136.0
Num trajectories: 1000
Cumulative reward per trajectory (first 10): [17.6581 21.3992 18.8986 20.8452 25.551  20.0527 17.9912 21.3489 21.7778
 17.7468]
Mean cumulative reward: 19.87270579624176
Std cumulative reward: 2.25881864128748
Min cumulative reward: 10.54045295715332
Max cumulative reward: 26.44003677368164

Inspecting PPO dataset:
Num trajectories: 1000
Cumulative cost per trajectory (first 10): [ 28.  74.  98.  97.  81. 124.  25.  31. 182.  36.]
Mean cumulative cost: 53.83
Std cumulative cost: 34.10174628959638
Min cumulative cost: 0.0
Max cumulative cost: 191.0
Num trajectories: 1000
Cumulative reward per trajectory (first 10): [21.4818 23.1181 24.5245 24.1004 23.4005 23.9068 21.2497 25.6249 22.8157
 23.8689]
Mean cum

In [ ]:
"""
Create a cost-balanced trajectory dataset from one or more pickled datasets.

Target composition by default:
- 700 trajectories with cumulative cost >= 25
- 700 trajectories with cumulative cost < 25

Examples:
python rudder/build_cost_balanced_trajectories.py \
  --inputs SafeDICE/dataset/safetygym/ppo_lagrangian_PointGoal1_s0.pickle \
           SafeDICE/dataset/safetygym/ppo_PointGoal1_s0.pickle \
  --output rudder/combined_cost_balanced_1400.pkl
"""

import argparse
import os
import pickle
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np


def _pick(mapping: Dict[str, Any], candidates: Sequence[str]) -> Optional[Any]:
    for key in candidates:
        if key in mapping:
            return mapping[key]
    return None


def _as_1d(arr: Any) -> np.ndarray:
    return np.asarray(arr).reshape(-1)


def _as_np32(arr: Any) -> np.ndarray:
    return np.asarray(arr, dtype=np.float32)


def _resolve_input_path(path: str) -> str:
    raw = os.path.expanduser(os.path.expandvars(path))
    normalized = raw.replace("\\", os.sep).replace("/", os.sep)

    candidates = []
    if os.path.isabs(normalized):
        candidates.append(normalized)
    else:
        candidates.append(os.path.abspath(normalized))
        repo_root = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
        candidates.append(os.path.abspath(os.path.join(repo_root, normalized)))

    for candidate in candidates:
        if os.path.isfile(candidate):
            return candidate

    raise FileNotFoundError(
        "Input dataset not found: %s. Tried: %s" % (path, ", ".join(candidates))
    )


def _extract_payload(loaded: Any) -> Any:
    payload = loaded
    if isinstance(payload, dict):
        # Common training checkpoint shells.
        ts = payload.get("training_state")
        if isinstance(ts, dict):
            for k in ("dataset", "data", "replay_buffer", "buffer", "expert_data", "trajectories"):
                if k in ts:
                    payload = ts[k]
                    break

        # Common top-level dataset shells.
        if payload is loaded:
            for k in ("dataset", "data", "expert_data", "trajectories"):
                if k in payload:
                    payload = payload[k]
                    break

    return payload


def _trajectory_cost(traj: Dict[str, Any]) -> float:
    costs = _pick(traj, ("costs", "cost", "c"))
    if costs is None:
        raise KeyError("Trajectory missing costs/cost/c field")
    return float(np.sum(_as_1d(costs).astype(np.float32)))


def _split_flat_dataset_into_trajectories(payload: Dict[str, Any]) -> List[Dict[str, np.ndarray]]:
    # Find canonical arrays.
    states = _pick(payload, ("states", "observations", "obs", "state", "s"))
    actions = _pick(payload, ("actions", "acts", "action", "a"))
    next_states = _pick(payload, ("next_states", "next_observations", "next_obs", "next_state", "s_next", "obs2"))
    rewards = _pick(payload, ("rewards", "reward", "rews", "r", "env_rewards"))
    costs = _pick(payload, ("costs", "cost", "c"))
    dones = _pick(payload, ("dones", "done", "terminals", "terminal", "episode_ends", "timeouts"))

    if costs is None:
        raise KeyError("No cost field found in flat dataset")
    if dones is None:
        raise KeyError("No done/terminal field found in flat dataset; cannot segment trajectories")

    # Keep only arrays that exist.
    arrays: Dict[str, np.ndarray] = {}
    if states is not None:
        arrays["states"] = _as_np32(states)
    if actions is not None:
        arrays["actions"] = _as_np32(actions)
    if next_states is not None:
        arrays["next_states"] = _as_np32(next_states)
    if rewards is not None:
        arrays["rewards"] = _as_np32(rewards)
    arrays["costs"] = _as_np32(costs)

    done_arr = _as_1d(dones).astype(bool)

    # Truncate all arrays to same length.
    lengths = [len(done_arr)] + [len(v) for v in arrays.values()]
    n = int(min(lengths))
    done_arr = done_arr[:n]
    for k in list(arrays.keys()):
        arrays[k] = arrays[k][:n]

    trajectories: List[Dict[str, np.ndarray]] = []
    start = 0
    for i, done in enumerate(done_arr):
        if done:
            if i + 1 > start:
                traj = {k: v[start:i + 1] for k, v in arrays.items()}
                traj["dones"] = done_arr[start:i + 1].astype(np.float32)
                trajectories.append(traj)
            start = i + 1

    if start < n:
        traj = {k: v[start:n] for k, v in arrays.items()}
        traj["dones"] = done_arr[start:n].astype(np.float32)
        trajectories.append(traj)

    return trajectories


def _extract_trajectories(loaded: Any) -> List[Dict[str, Any]]:
    payload = _extract_payload(loaded)

    if isinstance(payload, list):
        # Expected: list of trajectory dicts.
        trajs = [t for t in payload if isinstance(t, dict)]
        if not trajs:
            raise ValueError("Dataset payload is a list but contains no trajectory dicts")
        return trajs

    if isinstance(payload, dict):
        return _split_flat_dataset_into_trajectories(payload)

    raise TypeError("Unsupported dataset payload type: %s" % type(payload))


def _load_trajectories_from_paths(paths: Iterable[str]) -> List[Dict[str, Any]]:
    all_trajs: List[Dict[str, Any]] = []
    for p in paths:
        resolved = _resolve_input_path(p)
        with open(resolved, "rb") as f:
            loaded = pickle.load(f)
        trajs = _extract_trajectories(loaded)
        all_trajs.extend(trajs)
        print("Loaded %d trajectories from %s" % (len(trajs), resolved))
    return all_trajs


def _sample_balanced(
    trajectories: List[Dict[str, Any]],
    threshold: float,
    num_ge: int,
    num_lt: int,
    seed: int,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    ge_pool: List[Dict[str, Any]] = []
    lt_pool: List[Dict[str, Any]] = []

    for traj in trajectories:
        csum = _trajectory_cost(traj)
        if csum >= threshold:
            ge_pool.append(traj)
        else:
            lt_pool.append(traj)

    if len(ge_pool) < num_ge or len(lt_pool) < num_lt:
        raise ValueError(
            "Insufficient trajectories for requested split. "
            "Available >= threshold: %d (need %d), < threshold: %d (need %d)."
            % (len(ge_pool), num_ge, len(lt_pool), num_lt)
        )

    rng = np.random.RandomState(seed)
    ge_idx = rng.choice(len(ge_pool), size=num_ge, replace=False)
    lt_idx = rng.choice(len(lt_pool), size=num_lt, replace=False)

    selected_ge = [ge_pool[int(i)] for i in ge_idx]
    selected_lt = [lt_pool[int(i)] for i in lt_idx]

    combined = selected_ge + selected_lt
    rng.shuffle(combined)

    cost_sums = np.asarray([_trajectory_cost(t) for t in combined], dtype=np.float32)
    metadata = {
        "threshold": float(threshold),
        "requested_ge": int(num_ge),
        "requested_lt": int(num_lt),
        "selected_total": int(len(combined)),
        "selected_ge": int(np.sum(cost_sums >= threshold)),
        "selected_lt": int(np.sum(cost_sums < threshold)),
        "cost_sum_mean": float(cost_sums.mean()),
        "cost_sum_std": float(cost_sums.std()),
        "cost_sum_min": float(cost_sums.min()),
        "cost_sum_max": float(cost_sums.max()),
    }
    return combined, metadata


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Create a combined trajectory dataset balanced by cumulative cost"
    )
    parser.add_argument(
        "--inputs",
        type=str,
        nargs="+",
        required=True,
        help="One or more input pickle datasets",
    )
    parser.add_argument(
        "--output",
        type=str,
        required=True,
        help="Output pickle path",
    )
    parser.add_argument(
        "--threshold",
        type=float,
        default=25.0,
        help="Cumulative cost threshold",
    )
    parser.add_argument(
        "--num_ge",
        type=int,
        default=700,
        help="Number of trajectories with cumulative cost >= threshold",
    )
    parser.add_argument(
        "--num_lt",
        type=int,
        default=700,
        help="Number of trajectories with cumulative cost < threshold",
    )
    parser.add_argument("--seed", type=int, default=0, help="Random seed")
    args = parser.parse_args()

    trajectories = _load_trajectories_from_paths(args.inputs)
    print("Total trajectory pool: %d" % len(trajectories))

    combined, metadata = _sample_balanced(
        trajectories=trajectories,
        threshold=args.threshold,
        num_ge=args.num_ge,
        num_lt=args.num_lt,
        seed=args.seed,
    )

    out_path = os.path.expanduser(os.path.expandvars(args.output))
    out_path = out_path.replace("\\", os.sep).replace("/", os.sep)
    if not os.path.isabs(out_path):
        repo_root = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
        out_path = os.path.abspath(os.path.join(repo_root, out_path))

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    payload = {
        "trajectories": combined,
        "metadata": metadata,
    }
    with open(out_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    print("Saved combined dataset to %s" % out_path)
    print("Summary:")
    for k, v in metadata.items():
        print("  %s: %s" % (k, v))


if __name__ == "__main__":
    main()
